In [1]:
LAT_MIN, LAT_MAX = 28.4, 28.9
LON_MIN, LON_MAX = 76.8, 77.4

In [2]:
from pathlib import Path

# Get project root (2 levels up from this file)
BASE_DIR = Path().resolve()

In [3]:
print(BASE_DIR)

D:\17. Hackathon\Project\flood-analysis-project\src\data_loading


In [ ]:
RAW_PATH = BASE_DIR / "data" / "raw" / "rainfall" / "gpm" / "data"
PROCESSED_PATH = BASE_DIR / "data" / "processed" / "rainfall"

In [13]:
import xarray as xr
import pandas as pd
import glob

# Delhi bounds
LAT_MIN, LAT_MAX = 28.4, 28.9
LON_MIN, LON_MAX = 76.8, 77.4

# Get files
files = list(RAW_PATH.glob("*.nc4"))
print("Total files:", len(files))

dfs = []

for f in files:
    try:
        print("Processing:", f)

        # 🔥 Open with chunking (lazy loading = FAST)
        ds = xr.open_dataset(f)

        # 🔥 Select only required variable directly
        rain = ds['precipitation']

        # 🔥 Slice BEFORE loading into memory
        rain = rain.sel(
        #   lat=slice(LAT_MIN, LAT_MAX),
            lon=slice(LON_MIN, LON_MAX)
        )

        # 🔥 Drop NaNs early (reduces size)
        rain = rain.where(rain > 0, drop=True)

        # Convert to DataFrame
        df = rain.to_dataframe().reset_index()

        # Rename columns
        df.rename(columns={
            'precipitation': 'rain_gpm',
            'time': 'date'
        }, inplace=True)

        # Keep only needed columns
        df = df[['lat', 'lon', 'date', 'rain_gpm']]

        dfs.append(df)

    except Exception as e:
        print("Error in:", f, "|", e)

# 🔥 Concatenate ONLY ONCE (important)
if dfs:
    final_df = pd.concat(dfs, ignore_index=True)
    PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

    output_file = PROCESSED_PATH / "gpm_processed.csv"
    final_df.to_csv(output_file, index=False)

    print("Saved to:", output_file)    
    print("Final shape:", final_df.shape)
else:
    print("No data extracted")

Total files: 0
No data extracted
